In [11]:
import numpy as np

print("Numpy version:", np.__version__)

Numpy version: 2.5.4


In [ ]:
# Bagian A
# Fungsi Metrik Jarak

def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def manhattan_distance(a, b):
    return np.sum(np.abs(a - b))

def minkowski_distance(a, b, p):
    return np.sum(np.abs(a - b) ** p) ** (1 / p)

In [13]:
# data uji sederhana dengan 2 fitur
data_A = np.array([1, 2])
data_B = np.array([4, 6])

print("Euclidean:", euclidean_distance(data_A, data_B))
print("Manhattan:", manhattan_distance(data_A, data_B))
print("Minkowski (p=2):", minkowski_distance(data_A, data_B, p=2))

Euclidean: 5.0
Manhattan: 7
Minkowski (p=2): 5.0


In [ ]:
# bagian A
# Fungsi Normalisasi Data


def min_max_scaler(X_train, X_test):
    min_val = np.min(X_train, axis=0)
    max_val = np.max(X_train, axis=0)
    X_train_scaled = (X_train - min_val) / (max_val - min_val)
    X_test_scaled = (X_test - min_val) / (max_val - min_val)
    return X_train_scaled, X_test_scaled


def z_score_scaler(X_train, X_test):
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std
    return X_train_scaled, X_test_scaled

In [15]:
# Data uji untuk normalisasi

X_train = np.array([[10, 100], [20, 200]])
X_test = np.array([[15, 150]])

X_train_minmax, X_test_minmax = min_max_scaler(X_train, X_test)
print("MinMax Scaled Train:\n", X_train_minmax)
print("MinMax Scaled Test:\n", X_test_minmax)

MinMax Scaled Train:
 [[0. 0.]
 [1. 1.]]
MinMax Scaled Test:
 [[0.5 0.5]]


In [ ]:
# bagian A

import numpy as np

class KNNClassifier:
    def __init__(self, k=3, metric="euclidean", p=2, weights="uniform"):
        self.k = k
        self.metric = metric
        self.p = p
        self.weights = weights
        
    def fit(self, X, y):
        # Menyimpan data latih
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        
    def _distance(self, a, b):
        # Memanggil logika jarak
        if self.metric == "euclidean":
            return euclidean_distance(a, b)
        elif self.metric == "manhattan":
            return manhattan_distance(a, b)
        elif self.metric == "minkowski":
            return minkowski_distance(a, b, self.p)

    def predict(self, X_test):
        X_test = np.array(X_test)
        predictions = []
        
        for x in X_test:
            # Hitung jarak x ke setiap data di X_train
            distances = np.array([self._distance(x, x_tr) for x_tr in self.X_train])
            
            # Cari K indeks tetangga terdekat
            k_indices = np.argsort(distances)[:self.k]
            k_labels = self.y_train[k_indices]
            k_distances = distances[k_indices]
            
            # Voting (Uniform atau Distance)
            if self.weights == "uniform":
                # Cari kelas yang paling banyak muncul
                classes, counts = np.unique(k_labels, return_counts=True)
                max_count = np.max(counts)
                winning_classes = classes[counts == max_count]
                
                # Tie-Breaking Rule
                # Jika ada lebih dari 1 kelas pemenang, ambil kelas dari tetangga terdekat individual
                if len(winning_classes) > 1:
                    for label in k_labels:
                        if label in winning_classes:
                            final_pred = label
                            break
                else:
                    final_pred = winning_classes[0]
                    
            elif self.weights == "distance":
                # Hitung bobot = 1 / d^2
                # menghidari pembagian dengan nol dengan menambahkan epsilon kecil (1e-10)
                weights_val = 1.0 / (np.maximum(k_distances, 1e-10) ** 2)
                
                class_weights = {}
                for label, w in zip(k_labels, weights_val):
                    class_weights[label] = class_weights.get(label, 0.0) + w
                
                # Pilih kelas dengan total bobot terbesar
                final_pred = max(class_weights, key=class_weights.get)
                
            predictions.append(final_pred)
            
        return np.array(predictions)

In [20]:
# data latih studi kasus

X_train_studi = np.array([
    [85, 80], [60, 55], [75, 70], [50, 65],
    [90, 85], [65, 50], [70, 75], [55, 60]
])

y_train_studi = np.array([
    "Lulus", "Tidak Lulus", "Lulus", "Tidak Lulus",
    "Lulus", "Tidak Lulus", "Lulus", "Tidak Lulus"
])

# data uji mahasiswa x 
X_test_studi = np.array([[68, 64]])

# test k = 3, uniform
knn1 = KNNClassifier(k=3, metric="euclidean", weights="uniform")
knn1.fit(X_train_studi, y_train_studi)
print("Prediksi (k=3, uniform):", knn1.predict(X_test_studi))

# test k = 5, uniform
knn2 = KNNClassifier(k=5, metric="euclidean", weights="uniform")
knn2.fit(X_train_studi, y_train_studi)
print("Prediksi (k=5, uniform):", knn2.predict(X_test_studi))

# test k = 5, weighted
knn3 = KNNClassifier(k=5, metric="euclidean", weights="distance")
knn3.fit(X_train_studi, y_train_studi)
print("Prediksi (k=5, weighted):", knn3.predict(X_test_studi))


Prediksi (k=3, uniform): ['Lulus']
Prediksi (k=5, uniform): ['Tidak Lulus']
Prediksi (k=5, weighted): ['Lulus']


In [ ]:
# Bagian B


import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load dataset breast cancer
data = load_breast_cancer()
X = data.data
y = data.target

# bagi dataset menjadi 80% data latih dan 20% data uji
# nim digit terakhir = 07, random_state = 7
NIM_SEED = 7
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=NIM_SEED, stratify=y)

# Latih knn buatan sendiri dengan k=5, euclidean, uniform
knn_custom = KNNClassifier(k=5, metric="euclidean", weights="uniform")
knn_custom.fit(X_train, y_train)
y_predict_custom = knn_custom.predict(X_test)

# Latih knn sklearn dengan k=5, euclidean, uniform
knn_sklearn = KNeighborsClassifier(n_neighbors=5, metric="euclidean", weights="uniform")
knn_sklearn.fit(X_train, y_train)
y_predict_sklearn = knn_sklearn.predict(X_test)

# hitung presentase kesamaan prediksi
match_precentage = np.mean(y_predict_custom == y_predict_sklearn) * 100
print(f"Presentase kesamaan prediksi: {match_precentage:.2f}%")

# cek apakah ada data prediksi yang berbeda
defferences = np.where(y_predict_custom != y_predict_sklearn)[0]
if len(defferences) == 0:
    print("Semua prediksi sama antara knn buatan sendiri dan sklearn.")
else:
    print(f"Ada {len(defferences)} prediksi yang berbeda antara knn buatan sendiri dan sklearn.")
    print("Indeks data yang berbeda:", defferences)
    print("Prediksi knn buatan sendiri:", y_predict_custom[defferences])
    print("Prediksi knn sklearn:", y_predict_sklearn[defferences])


Presentase kesamaan prediksi: 100.00%
Semua prediksi sama antara knn buatan sendiri dan sklearn.


In [ ]:
# Bagian C